# Семинар 9 - Методы построения оптического потока по последовательности изображений

**Этот семинар содержит оцениваемое домашнее задание**

***

Источник - https://habr.com/ru/post/201406/

$\textbf{Task statement}$: Оптический поток (ОП) – изображение видимого движения, представляющее собой сдвиг каждой точки (пикселя) между двумя изображениями.

По сути, он представляет собой поле скоростей. Суть ОП в том, что для каждой точки изображения $I_{t_0} (\vec{r})$ находится такой вектор сдвига $\delta \vec{r}$, чтобы было соответсвие между исходной точкой и точкой на следущем фрейме $I_{t_1} (\vec{r} + \delta \vec{r})$. В качестве метрики соответвия берут близость интенсивности пикселей, беря во внимание маленькую разницу по времени между кадрами: $\delta{t} = t_{1} - t_{0}$. В более точных методах точку можно привязывать к объекту на основе, например, выделения ключевых точек, а также считать градиенты вокруг точки, лапласианы и проч.

$\textbf{For what}$: Определение собственной скорости, Определение локализации, Улучшение методов трекинга объектов, сегментации, Детектирование событий, Сжатие видеопотока и проч.

![](data/tennis.png)

Разделяют 2 вида оптического потока - плотный (dense) [Farneback method, neural nets], работающий с целым изображением, и выборочный (sparse) [Lucas-Kanade method], работающий с ключевыми точками

In [ ]:
# !wget https://www.bogotobogo.com/python/OpenCV_Python/images/mean_shift_tracking/slow_traffic_small.mp4 -O data/slow_traffic_small.mp4

In [1]:
import cv2
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import IPython

%matplotlib inline

## Lucas-Kanade (sparse)

Пусть $I_{1} = I(x, y, t_{1})$ интенсивность в некоторой точке (x, y) на первом изображении (т. е. в момент времени t). На втором изображении эта точка сдвинулась на (dx, dy), при этом прошло время dt, тогда $I_{2} = I(x + dx, y + dx, t_{1} + dt) \approx I_{1} + I_{x}dx + I_{y}dy +  I_{t}dt$. Из постановки задачи следует, что интенсивность пикселя не изменилась, тогда $I_{1} = I_{2}$. Далее определяем $dx, dy$.

Самое простое решение проблемы – алгоритм Лукаса-Канаде. У нас же на изображении объекты размером больше 1 пикселя, значит, скорее всего, в окрестности текущей точки у других точек будут примерно такие же сдвиги. Поэтому мы возьмем окно вокруг этой точки и минимизируем (по МНК) в нем суммарную погрешность с весовыми коэффициентами, распределенными по Гауссу, то есть так, чтобы наибольший вес имели пиксели, ближе всего находящиеся к исследуемому.

**Полезные материалы:** 
- цикл видео-лекций от First Principles of Computer Vision, посвященный Optical Flow и алгоритму Lucas-Kanade: https://youtube.com/playlist?list=PL2zRqk16wsdoYzrWStffqBAoUY8XdvatV

### Вопрос 1

Перечислите три основных предположения, на которых базируется метод Lucas-Kanade. Почему каждое из них важно для корректной работы алгоритма?

**Ответ:**

1. Яркость сохраняется

Для одной и той же точки сцены интенсивность  меняется незначительно между кадрами. Важно так как это дает основное уравнение оптического потока

2. Малое смещение между соседними кадрами

Сдвиг должен быть небольшим чтобы линейное разложение Тейлора было корректным. Иначе аппроксимация I(x+u,y+v) через градиенты становится неточной.

3. Локально постоянный поток в окне

В небольшой окрестности точки все пиксели движутся примерно одинаково. Это позволяет устойчиво оценить движение.


### Вопрос 2

Объясните, зачем нужен пирамидальный подход в алгоритме Lucas-Kanade. Какую проблему он решает и как именно?

**Ответ:**

Пирамидальный подход нужен, чтобы обрабатывать большие смещения. Сначала оценка на грубом (уменьшенном) уровне, где тот же физический сдвиг в пикселях становится меньше и малое смещение снова работает. Затем оценка последовательно уточняется на более детальных уровнях.

### Вопрос 3

С какими проблемами может столкнуться алгоритм Lucas-Kanade при отслеживании точек на видео? Назовите минимум три ограничения.

**Ответ:**


1. На однородных областях или одномерных текстурах движение неоднозначно.

2. Изменение яркости/освещения нарушает предположение сохранения интенсивности.

3. Шум, blur, компрессия ухудшают градиенты и стабильность решения.


### Задание 1

Напишите реализацию Лукаса-Канаде c помощью numpy и cv2. Сравните с реализацией `cv2.calcOpticalFlowPyrLK`.

In [2]:
import cv2
import numpy as np

def build_image_pyramid(image, num_levels, scale_factor=0.5):
    """
    Создаёт пирамиду изображений с уменьшающимся разрешением.
    """
    pyramid = [image.copy()]
    for _ in range(1, num_levels):
        prev = pyramid[-1]
        new_w = int(prev.shape[1] * scale_factor)
        new_h = int(prev.shape[0] * scale_factor)
        down = cv2.resize(prev, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
        pyramid.append(down)
    return pyramid


def compute_image_gradients(image):
    """
    Вычисляет пространственные градиенты изображения.
    """
    Ix = cv2.Sobel(image, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy


def compute_lk_optical_flow_point(Ix, Iy, It, window_size=5):
    """
    Вычисляет оптический поток по методу Lucas-Kanade для одного окна.
    """
    half = window_size // 2
    # Координаты центра окна
    h, w = Ix.shape
    center_x = w // 2
    center_y = h // 2

    # Определяем границы окна
    y1 = max(0, center_y - half)
    y2 = min(h, center_y + half + 1)
    x1 = max(0, center_x - half)
    x2 = min(w, center_x + half + 1)

    Ix_win = Ix[y1:y2, x1:x2].flatten()
    Iy_win = Iy[y1:y2, x1:x2].flatten()
    It_win = It[y1:y2, x1:x2].flatten()

    # Формируем матрицу A и вектор b
    A = np.array([[np.sum(Ix_win * Ix_win), np.sum(Ix_win * Iy_win)],
                  [np.sum(Ix_win * Iy_win), np.sum(Iy_win * Iy_win)]])
    b = np.array([-np.sum(Ix_win * It_win), -np.sum(Iy_win * It_win)])

    # Проверяем обусловленность
    eigvals = np.linalg.eigvals(A)
    if min(eigvals) < 1e-4:
        return None, None

    try:
        u, v = np.linalg.solve(A, b)
        return u, v
    except np.linalg.LinAlgError:
        return None, None


def compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size=5):
    """
    Вычисляет оптический поток для патча изображения.
    """
    Ix, Iy = compute_image_gradients(prev_patch)
    It = curr_patch - prev_patch
    u, v = compute_lk_optical_flow_point(Ix, Iy, It, window_size)
    return u, v


def track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, point, window_size=15, max_iterations=10, epsilon=0.01):
    """
    Отслеживает точку между кадрами с использованием пирамидального LK.
    """
    # Начальное смещение
    dx, dy = 0.0, 0.0
    # Текущая позиция точки на уровне пирамиды
    x, y = float(point[0]), float(point[1])

    # Количество уровней
    levels = len(prev_pyramid)

    # Проход сверху вниз
    for level in range(levels-1, -1, -1):
        # Масштаб текущего уровня
        scale = 1.0 / (2.0 ** level)

        # Текущая позиция точки на этом уровне пирамиды
        x_level = x * scale
        y_level = y * scale

        # Обновляем смещение: переносим с верхнего уровня
        dx = dx * 2.0
        dy = dy * 2.0

        # Изображения текущего уровня
        prev_img = prev_pyramid[level]
        curr_img = curr_pyramid[level]

        # Итеративное уточнение
        for _ in range(max_iterations):
            # Координаты окна
            half = window_size // 2
            x0 = int(round(x_level + dx))
            y0 = int(round(y_level + dy))

            # Проверка границ
            h, w = prev_img.shape
            if (x0 - half < 0) or (x0 + half >= w) or (y0 - half < 0) or (y0 + half >= h):
                return None

            # Извлекаем патчи
            prev_patch = prev_img[y0-half:y0+half+1, x0-half:x0+half+1].astype(np.float32)
            curr_patch = curr_img[y0-half:y0+half+1, x0-half:x0+half+1].astype(np.float32)

            # Вычисляем оптический поток для патча
            u, v = compute_lk_optical_flow_for_patch(prev_patch, curr_patch, window_size)
            if u is None or v is None:
                return None

            # Обновляем смещение
            dx += u
            dy += v

            # Проверка сходимости
            if u*u + v*v < epsilon*epsilon:
                break

        # Сохраняем финальное смещение для этого уровня
    # Финальная позиция
    final_x = x + dx
    final_y = y + dy
    return (final_x, final_y)


def lucas_kanade_optical_flow(prev_frame, curr_frame, points,
                             window_size=15, num_pyramid_levels=3,
                             max_iterations=10, epsilon=0.01):
    """
    Вычисляет разреженный оптический поток методом Лукаса-Канаде.
    """
    # Преобразуем в grayscale и нормализуем
    if len(prev_frame.shape) == 3:
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    else:
        prev_gray = prev_frame.astype(np.float32) / 255.0

    if len(curr_frame.shape) == 3:
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    else:
        curr_gray = curr_frame.astype(np.float32) / 255.0

    # Строим пирамиды
    prev_pyramid = build_image_pyramid(prev_gray, num_pyramid_levels)
    curr_pyramid = build_image_pyramid(curr_gray, num_pyramid_levels)

    new_points = []
    status = []

    for pt in points:
        x, y = pt
        # Убеждаемся, что точка в границах исходного изображения
        if x < 0 or y < 0 or x >= prev_gray.shape[1] or y >= prev_gray.shape[0]:
            new_points.append([x, y])
            status.append(0)
            continue

        tracked = track_point_with_pyramid_lk(prev_pyramid, curr_pyramid, (x, y),
                                              window_size, max_iterations, epsilon)
        if tracked is None:
            new_points.append([x, y])
            status.append(0)
        else:
            new_points.append([tracked[0], tracked[1]])
            status.append(1)

    return np.array(new_points, dtype=np.float32), np.array(status, dtype=np.uint8)


def demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео.
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
    p0 = p0.reshape(-1, 2)  # Преобразуем в формат [[x1, y1], [x2, y2], ...]

    # Сохраняем изначальные точки для отслеживания через все видео
    initial_points = p0.copy()

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (len(p0), 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью нашей реализации
        p1, st = lucas_kanade_optical_flow(
            old_gray,
            frame_gray,
            p0,
            window_size=15,
            num_pyramid_levels=3
        )

        # Выбираем хорошие точки
        good_new = p1[st == 1]
        good_old = p0[st == 1]

        # Рисуем треки
        for idx, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[idx % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[idx % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0[st == 1] = good_new

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [3]:
result_path = demo_optical_flow(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [00:13<00:00, 70.02it/s] 

Результат сохранен в output_my_LK.mp4


### Релизация OpenCV - cv2.calcOpticalFlowPyrLK

In [4]:
def demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_my_LK.mp4'):
    """
    Демонстрация работы алгоритма на видео с использованием cv2.calcOpticalFlowPyrLK.

    Args:
        video_path: Путь к входному видео
        output_path: Путь для сохранения результата
    """
    # Открываем видео
    cap = cv2.VideoCapture(video_path)

    # Получаем параметры видео
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Настраиваем запись выходного видео
    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Параметры для обнаружения углов Shi-Tomasi
    feature_params = dict(
        maxCorners=100,
        qualityLevel=0.3,
        minDistance=7,
        blockSize=7
    )

    # Параметры для Lucas-Kanade оптического потока
    lk_params = dict(
        winSize=(15, 15),
        maxLevel=3,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
    )

    # Берем первый кадр и находим в нем углы
    ret, old_frame = cap.read()
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    # Создаем маску для рисования
    mask = np.zeros_like(old_frame)

    # Создаем случайные цвета для визуализации
    color = np.random.randint(0, 255, (100, 3))

    from tqdm import tqdm
    for i in tqdm(range(length - 1)):  # -1 потому что первый кадр мы уже прочитали
        ret, frame = cap.read()

        if not ret:
            print('No frames grabbed!')
            break

        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Вычисляем оптический поток с помощью встроенной функции cv2.calcOpticalFlowPyrLK
        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        # Выбираем хорошие точки
        if p1 is not None:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

        # Рисуем треки
        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new.ravel()
            c, d = old.ravel()
            mask = cv2.line(mask, (int(a), int(b)), (int(c), int(d)), color[i % len(color)].tolist(), 2)
            frame = cv2.circle(frame, (int(a), int(b)), 5, color[i % len(color)].tolist(), -1)

        # Объединяем кадр и маску
        img = cv2.add(frame, mask)

        # Записываем результат
        out.write(img)

        # Обновляем предыдущий кадр
        old_gray = frame_gray.copy()

        # Обновляем точки, но только те, которые успешно отслежены
        p0 = good_new.reshape(-1, 1, 2)

    # Освобождаем ресурсы
    cap.release()
    out.release()

    print(f"Результат сохранен в {output_path}")
    return output_path

In [5]:
result_path = demo_optical_flow_opencv(video_path='data/slow_traffic_small.mp4', output_path='output_opencv_LK.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [00:01<00:00, 514.66it/s]

Результат сохранен в output_opencv_LK.mp4


### Задание 2

В базовой реализации у кода есть одна важная проблема - ключевые точки инициализируются единожды. В реальных задачах необходимо отслеживать точки, которые исчезают из кадра и появляются в других местах. Реализуйте механизм, который будет отслеживать точки, которые пропадают из кадра и добавлять новые точки в те места, где они появляются. Для этого вам нужно будет реализовать механизм поиска новых точек на изображении.

In [9]:
import cv2
import numpy as np
from tqdm import tqdm

# ------------------------------------------------------------
# Вспомогательные функции для Lucas-Kanade (пирамидальный LK)
# ------------------------------------------------------------

def build_pyr(img, levels, factor=0.5):
    pyr = [img.copy()]
    for _ in range(1, levels):
        prev = pyr[-1]
        h, w = prev.shape
        pyr.append(cv2.resize(prev, (int(w*factor), int(h*factor)), cv2.INTER_LINEAR))
    return pyr

def grad(I):
    Ix = cv2.Sobel(I, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(I, cv2.CV_64F, 0, 1, ksize=3)
    return Ix, Iy

def lk_point(Ix, Iy, It, win=5):
    hw = win//2
    h, w = Ix.shape
    cx, cy = w//2, h//2
    y1, y2 = max(0, cy-hw), min(h, cy+hw+1)
    x1, x2 = max(0, cx-hw), min(w, cx+hw+1)
    Ixw = Ix[y1:y2, x1:x2].flatten()
    Iyw = Iy[y1:y2, x1:x2].flatten()
    Itw = It[y1:y2, x1:x2].flatten()
    A = np.array([[np.sum(Ixw*Ixw), np.sum(Ixw*Iyw)],
                  [np.sum(Ixw*Iyw), np.sum(Iyw*Iyw)]])
    b = np.array([-np.sum(Ixw*Itw), -np.sum(Iyw*Itw)])
    if min(np.linalg.eigvals(A)) < 1e-4:
        return None, None
    try:
        u, v = np.linalg.solve(A, b)
        return u, v
    except:
        return None, None

def lk_patch(prev, curr, win=5):
    Ix, Iy = grad(prev)
    It = curr - prev
    return lk_point(Ix, Iy, It, win)

def track_pyr(prev_pyr, curr_pyr, pt, win=15, iters=10, eps=0.01):
    dx, dy = 0.0, 0.0
    x, y = float(pt[0]), float(pt[1])
    L = len(prev_pyr)
    for lvl in range(L-1, -1, -1):
        scale = 1.0 / (2**lvl)
        xl, yl = x*scale, y*scale
        dx *= 2.0
        dy *= 2.0
        Iprev = prev_pyr[lvl]
        Icurr = curr_pyr[lvl]
        for _ in range(iters):
            hw = win//2
            x0 = int(round(xl + dx))
            y0 = int(round(yl + dy))
            h, w = Iprev.shape
            if x0-hw < 0 or x0+hw >= w or y0-hw < 0 or y0+hw >= h:
                return None
            p1 = Iprev[y0-hw:y0+hw+1, x0-hw:x0+hw+1].astype(np.float32)
            p2 = Icurr[y0-hw:y0+hw+1, x0-hw:x0+hw+1].astype(np.float32)
            u, v = lk_patch(p1, p2, win)
            if u is None:
                return None
            dx += u
            dy += v
            if u*u + v*v < eps*eps:
                break
    return (x + dx, y + dy)

def lucas_kanade_optical_flow(prev, curr, pts, window_size=15, num_pyramid_levels=3,
                              max_iterations=10, epsilon=0.01):
    # Приведение к grayscale и нормализация
    if len(prev.shape) == 3:
        prev = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY).astype(np.float32)/255.0
    else:
        prev = prev.astype(np.float32)/255.0
    if len(curr.shape) == 3:
        curr = cv2.cvtColor(curr, cv2.COLOR_BGR2GRAY).astype(np.float32)/255.0
    else:
        curr = curr.astype(np.float32)/255.0

    pyr_prev = build_pyr(prev, num_pyramid_levels)
    pyr_curr = build_pyr(curr, num_pyramid_levels)

    new_pts = []
    status = []
    for pt in pts:
        x, y = pt
        if x<0 or y<0 or x>=prev.shape[1] or y>=prev.shape[0]:
            new_pts.append([x, y]); status.append(0); continue
        tracked = track_pyr(pyr_prev, pyr_curr, (x, y), window_size, max_iterations, epsilon)
        if tracked is None:
            new_pts.append([x, y]); status.append(0)
        else:
            new_pts.append([tracked[0], tracked[1]]); status.append(1)
    return np.array(new_pts, dtype=np.float32), np.array(status, dtype=np.uint8)

# ------------------------------------------------------------
# Ваши функции detect_new_points, merge_points, demo...
# ------------------------------------------------------------

def detect_new_points(frame_gray, existing_points, max_corners=120, quality_level=0.3,
                      min_distance=7, block_size=7):
    h, w = frame_gray.shape[:2]
    mask = np.full((h, w), 255, dtype=np.uint8)
    if existing_points is not None and len(existing_points) > 0:
        for x, y in existing_points:
            cv2.circle(mask, (int(x), int(y)), int(min_distance * 1.5), 0, -1)
    new_pts = cv2.goodFeaturesToTrack(frame_gray, mask=mask, maxCorners=max_corners,
                                      qualityLevel=quality_level, minDistance=min_distance,
                                      blockSize=block_size)
    if new_pts is None:
        return np.empty((0, 2), dtype=np.float32)
    return new_pts.reshape(-1, 2).astype(np.float32)

def merge_points(old_points, tracked_points, status, frame_gray, target_points=120):
    if tracked_points is None or len(tracked_points) == 0:
        alive = np.empty((0, 2), dtype=np.float32)
    else:
        alive = tracked_points[status == 1].astype(np.float32)
    need = max(0, target_points - len(alive))
    if need > 0:
        new_pts = detect_new_points(frame_gray, alive, max_corners=need,
                                    quality_level=0.3, min_distance=7, block_size=7)
        if len(new_pts) > 0:
            alive = np.vstack([alive, new_pts]) if len(alive) > 0 else new_pts
    return alive.astype(np.float32)

def demo_optical_flow_with_point_recovery(video_path='data/slow_traffic_small.mp4',
                                          output_path='output_my_LK_recovery.mp4',
                                          target_points=120):
    cap = cv2.VideoCapture(video_path)
    length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*'MP4V')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    ret, old_frame = cap.read()
    if not ret:
        cap.release(); out.release()
        raise RuntimeError("Не удалось прочитать первый кадр")

    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
    p0 = detect_new_points(old_gray, existing_points=None, max_corners=target_points)

    mask_tracks = np.zeros_like(old_frame)
    color = np.random.randint(0, 255, (2000, 3), dtype=np.uint8)

    for _ in tqdm(range(length - 1)):
        ret, frame = cap.read()
        if not ret:
            break
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if p0 is None or len(p0) == 0:
            p0 = detect_new_points(frame_gray, None, max_corners=target_points)
            old_gray = frame_gray.copy()
            out.write(frame)
            continue

        p1, st = lucas_kanade_optical_flow(old_gray, frame_gray, p0,
                                           window_size=15, num_pyramid_levels=3,
                                           max_iterations=10, epsilon=0.01)

        good_new = p1[st == 1]
        good_old = p0[st == 1]

        for i, (new, old) in enumerate(zip(good_new, good_old)):
            a, b = new
            c, d = old
            col = color[i % len(color)].tolist()
            mask_tracks = cv2.line(mask_tracks, (int(c), int(d)), (int(a), int(b)), col, 2)
            frame = cv2.circle(frame, (int(a), int(b)), 4, col, -1)

        p0 = merge_points(p0, p1, st, frame_gray, target_points=target_points)
        img = cv2.add(frame, mask_tracks)
        out.write(img)
        old_gray = frame_gray.copy()

    cap.release()
    out.release()
    print(f"Результат сохранен в {output_path}")
    return output_path

# ------------------------------------------------------------
# Запуск
# ------------------------------------------------------------
if __name__ == '__main__':
    result_path = demo_optical_flow_with_point_recovery(
        video_path='data/slow_traffic_small.mp4',
        output_path='output_my_LK_recovery.mp4',
        target_points=120
    )
    print("Result:", result_path)

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [02:02<00:00,  7.48it/s]

Результат сохранен в output_my_LK_recovery.mp4
Result: output_my_LK_recovery.mp4


### Вопрос 4

В чем основное отличие разреженного (sparse) оптического потока Lucas-Kanade от плотного (dense) оптического потока (например, метода Farneback)?

**Ответ:**

1. Sparse LK: считает поток только для выбранных ключевых точек (углы/фичи), быстрее и дешевле, удобен для трекинга признаков.

2. Dense (Farneback): оценивает вектор движения для каждого пикселя, дает полное поле потока, но обычно тяжелее по вычислениям и памяти.

## Farneback (dense)

Метод Farneback носит несколько более глобальный характер, чем метод Лукаса-Канаде. Он опирается на предположение о том, что на всем изображении оптический поток будет достаточно гладким.

# Вопрос 5

Перечислите основные шаги алгоритма Farneback для расчета оптического потока.

**Ответ:**
1. Построение гауссовой пирамиды кадров

2. В каждой локальной области аппроксимация сигнала квадратичным полиномом

3. Сопоставление полиномиальных моделей двух кадров для оценки локального смещения

4. Итеративное уточнение потока на каждом уровне

5. Переход от грубого уровня к более детальному с распространением оценки

6. Получение плотного поля (u,v) для всех пикселей


### Вопрос 6

Каким образом в методе Farneback обрабатываются большие смещения объектов между кадрами?

**Ответ:**
Большие смещения в Farneback обрабатываются за счет пирамидального подхода:

- на верхних (уменьшенных) уровнях большие реальные движения становятся малыми в пикселях;

- сначала оценивается грубый поток;

- затем он масштабируется вниз и уточняется на более детальных уровнях


In [10]:
import cv2
import numpy as np
from tqdm import tqdm

def demo_fb(vid='data/slow_traffic_small.mp4', out='output_opencv_farneback.mp4'):
    cap = cv2.VideoCapture(vid)
    l = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    out_v = cv2.VideoWriter(out, cv2.VideoWriter_fourcc(*'MP4V'), fps, (w, h))

    ret, f1 = cap.read()
    if not ret:
        print('no video')
        return None
    prvs = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY)
    hsv = np.zeros_like(f1)
    hsv[..., 1] = 255

    for _ in tqdm(range(l-1)):
        ret, f2 = cap.read()
        if not ret:
            break
        nxt = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY)
        flow = cv2.calcOpticalFlowFarneback(prvs, nxt, None, 0.5, 3, 15, 3, 5, 1.2, 0)
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        hsv[..., 0] = ang * 180 / np.pi / 2
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
        out_v.write(bgr)
        prvs = nxt

    cap.release()
    out_v.release()
    print(f'done: {out}')
    return out

result_path = demo_fb(vid='data/slow_traffic_small.mp4', out='output_opencv_farneback.mp4')

OpenCV: FFMPEG: tag 0x5634504d/'MP4V' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|██████████| 913/913 [00:39<00:00, 23.24it/s]

done: output_opencv_farneback.mp4


### Вопрос 7

Как влияет предварительная обработка изображений (фильтрация шума, выравнивание гистограмм) на качество оптического потока, получаемого методом Farneback? Предложите оптимальный пайплайн предобработки.

**Ответ:**
